In [9]:
# ------------------------------------------------------------
# CELL 1 — Imports
# ------------------------------------------------------------
import requests
import psycopg2
from datetime import datetime, timezone, timedelta
import json

print("Libraries loaded.")

Libraries loaded.


In [10]:
# ------------------------------------------------------------
# CELL 1.5 — Secure credential input
# ------------------------------------------------------------
from getpass import getpass

DB_PASSWORD = getpass("Enter your PostgreSQL password: ")

Enter your PostgreSQL password:  ········


In [11]:
# ------------------------------------------------------------
# CELL 2 — Configuration
# ------------------------------------------------------------

# --- SNOTEL Station ---
# Station triplet format: "<STATION_ID>:<STATE_ABBR>:SNTL"
# https://wcc.sc.egov.usda.gov/nwcc/yearcount?network=sntl&state=&counttype=listwithdiscontinued
STATION_TRIPLET = "846:CA:SNTL"
STATION_ID      = "846"
STATION_NAME    = "Virginia Lakes Ridge"  # confirm this when you verify below

# --- AWDB REST API ---
AWDB_BASE_URL = "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1"

# Elements to pull
ELEMENT_CODES = [
    "SNWD",   # Snow depth (inches)
    "WTEQ",   # Snow water equivalent (inches)
    "TAVG",   # Average air temperature (°F)
    "PREC",   # Precipitation accumulation (inches)
    "TMAX",   # Max air temperature (°F)
    "TMIN",   # Min air temperature (°F)
]

# --- DB connection ---
DB_SETTINGS = {
    "host":     "localhost",
    "database": "gis_portfolio",
    "user":     "postgres",
    "password": DB_PASSWORD,
    "port":     5432
}

print(f"Config set — targeting station: {STATION_TRIPLET}")

Config set — targeting station: 846:CA:SNTL


In [12]:
# -----------------------
# CELL 2.5: Station Test
# -----------------------
def inspect_station(station_triplet: str):
    """
    Hits the AWDB /stations metadata endpoint to confirm the
    station exists and prints all available element codes.
    """
    endpoint = f"{AWDB_BASE_URL}/stations"
    params = {
        "stationTriplets": station_triplet,
        "returnStationElements": "true",   # include what elements it measures
    }

    print(f"Looking up station: {station_triplet}")
    r = requests.get(endpoint, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if not data:
        print("Station not found — double-check your triplet.")
        return

    station = data[0]
    print(f"\nStation found!")
    print(f"   Name      : {station.get('name')}")
    print(f"   State     : {station.get('state')}")
    print(f"   Elevation : {station.get('elevation')} ft")
    print(f"   Lat/Lon   : {station.get('latitude')}, {station.get('longitude')}")
    print(f"   Active    : {station.get('activeFlag')}")

    print(f"\nAvailable element codes:")
    for el in station.get("stationElements", []):
        code     = el.get("elementCode", "?")
        duration = el.get("duration", "?")
        unit     = el.get("storedUnitCode", "?")
        print(f"   {code:<10} duration={duration:<8} unit={unit}")
inspect_station("846:CA:SNTL")



Looking up station: 846:CA:SNTL

Station found!
   Name      : Virginia Lakes Ridge
   State     : None
   Elevation : 9400.0 ft
   Lat/Lon   : 38.07298, -119.23433
   Active    : None

Available element codes:
   BATT       duration=?        unit=volt
   BATT       duration=?        unit=volt
   PRCP       duration=?        unit=in
   PRCP       duration=?        unit=in
   PRCP       duration=?        unit=in
   PRCP       duration=?        unit=in
   PRCP       duration=?        unit=in
   PRCPMTD    duration=?        unit=in
   PRCPSA     duration=?        unit=in
   PRCPSA     duration=?        unit=in
   PRCPSA     duration=?        unit=in
   PRCPSA     duration=?        unit=in
   PRCPSA     duration=?        unit=in
   PREC       duration=?        unit=in
   PREC       duration=?        unit=in
   PREC       duration=?        unit=in
   PREC       duration=?        unit=in
   PREC       duration=?        unit=in
   PREC       duration=?        unit=in
   RDC        duration=? 

In [13]:
# ------------------------------------------------------------
# CELL 3 — Fetch latest hourly data from AWDB REST API
# ------------------------------------------------------------

def fetch_snotel_latest(station_triplet: str, element_codes: list) -> dict:

    endpoint = f"{AWDB_BASE_URL}/data"
    now_utc    = datetime.now(timezone.utc)
    begin_date = (now_utc - timedelta(days=2)).strftime("%Y-%m-%d")
    end_date   = now_utc.strftime("%Y-%m-%d")

    # KEY FIX: REST API uses 'elements' (not 'elementCd') and 'duration' (not 'duration')
    # Multiple elements = comma-separated string in ONE 'elements' param
    params = {
        "stationTriplets": station_triplet,
        "elements":        ",".join(element_codes),
        "duration":        "HOURLY",
        "beginDate":       begin_date,
        "endDate":         end_date,
        "periodRef":       "START",
        "centralTendencyType": "NONE",
    }

    print(f"🌐 Requesting AWDB data for station {station_triplet}...")
    print(f"   Date range : {begin_date} → {end_date}")
    print(f"   Elements   : {params['elements']}")

    try:
        response = requests.get(endpoint, params=params, timeout=30)
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}\nResponse body: {response.text}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return None

    raw = response.json()
    print(f"API response received.")

    if not raw or not isinstance(raw, list):
        print(f"Unexpected response: {raw}")
        return None

    station_data = raw[0]

    # Map element codes -> DB column names
    element_map = {
        "SNWD":  "snow_depth_in",
        "WTEQ":  "swe_in",
        "TAVG":  "temp_avg_f",
        "PREC":  "precip_accum_in",
        "TMAX":  "temp_max_f",
        "TMIN":  "temp_min_f",
    }

    result = {
        "station_id":   station_triplet.split(":")[0],
        "capture_time": now_utc,
    }

    for element_data in station_data.get("data", []):
        code   = element_data.get("stationElement", {}).get("elementCode", "")
        values = element_data.get("values", [])

        # Walk backwards to find most recent non-null value
        latest_val = None
        latest_ts  = None
        for entry in reversed(values):
            v = entry.get("value")
            if v is not None:
                try:
                    latest_val = float(v)
                    latest_ts  = entry.get("date") or entry.get("dateTime")
                except (ValueError, TypeError):
                    pass
                break

        col_name = element_map.get(code)
        if col_name:
            result[col_name] = latest_val
            print(f"   {col_name:20s} = {latest_val}  (as of {latest_ts})")
        else:
            print(f"   (unmapped element: {code} = {latest_val})")

    return result


telemetry_data = fetch_snotel_latest(STATION_TRIPLET, ELEMENT_CODES)

if telemetry_data:
    print("\n Final telemetry_data dict:")
    for k, v in telemetry_data.items():
        print(f"   {k}: {v}")
else:
    print("No telemetry data returned.")

🌐 Requesting AWDB data for station 846:CA:SNTL...
   Date range : 2026-05-29 → 2026-05-31
   Elements   : SNWD,WTEQ,TAVG,PREC,TMAX,TMIN
API response received.
   precip_accum_in      = 24.9  (as of 2026-05-30 17:00)
   snow_depth_in        = 0.0  (as of 2026-05-30 17:00)
   swe_in               = 0.1  (as of 2026-05-30 17:00)

 Final telemetry_data dict:
   station_id: 846
   capture_time: 2026-05-31 01:48:16.937716+00:00
   precip_accum_in: 24.9
   snow_depth_in: 0.0
   swe_in: 0.1


In [14]:
# ------------------------------------------------------------
# CELL 4 — Inspect raw JSON (debug)
# ------------------------------------------------------------

import requests, json
from datetime import datetime, timezone, timedelta

now_utc = datetime.now(timezone.utc)

debug_params = [
    ("stationTriplets", "846:CA:SNTL"),   # was 1050
    ("duration",        "HOURLY"),
    ("beginDate",       (now_utc - timedelta(days=3)).strftime("%Y-%m-%d")),
    ("endDate",         now_utc.strftime("%Y-%m-%d")),
    ("periodRef",       "START"),
    ("centralTendencyType", "NONE"),
    ("elements",        "SNWD"),
    ("elements",        "WTEQ"),
]

r = requests.get(
    "https://wcc.sc.egov.usda.gov/awdbRestApi/services/v1/data",
    params=debug_params,
    timeout=30
)
print(f"Status: {r.status_code}")
print(json.dumps(r.json(), indent=2))

Status: 200
[
  {
    "stationTriplet": "846:CA:SNTL",
    "data": [
      {
        "stationElement": {
          "elementCode": "SNWD",
          "ordinal": 1,
          "durationName": "HOURLY",
          "dataPrecision": 0,
          "storedUnitCode": "in",
          "originalUnitCode": "in",
          "beginDate": "2001-08-02 09:00",
          "endDate": "2100-01-01 00:00",
          "derivedData": false
        },
        "values": [
          {
            "date": "2026-05-28 00:00",
            "value": 0
          },
          {
            "date": "2026-05-28 01:00",
            "value": 0
          },
          {
            "date": "2026-05-28 02:00",
            "value": 0
          },
          {
            "date": "2026-05-28 03:00",
            "value": 0
          },
          {
            "date": "2026-05-28 04:00",
            "value": 0
          },
          {
            "date": "2026-05-28 05:00",
            "value": 0
          },
          {
            "dat

In [15]:
# ------------------------------------------------------------
# CELL 5 — Insert into PostGIS DB
# ------------------------------------------------------------

if telemetry_data is None:
    print("Skipping DB insert — no telemetry data available.")
else:
    try:
        print("Connecting to local PostgreSQL database...")

        with psycopg2.connect(**DB_SETTINGS) as conn:
            with conn.cursor() as cur:

                insert_query = """
                    INSERT INTO "skiGIS".weather_logs (
                        station_id,
                        capture_time,
                        snow_depth_in,
                        swe_in,
                        precip_accum_in
                    )
                    VALUES (%s, %s, %s, %s, %s)
                    ON CONFLICT DO NOTHING;
                """

                cur.execute(insert_query, (
                    telemetry_data["station_id"],
                    telemetry_data["capture_time"],
                    telemetry_data.get("snow_depth_in"),
                    telemetry_data.get("swe_in"),
                    telemetry_data.get("precip_accum_in"),
                ))

        print("Ingest Layer Success: Transactional insert complete.")

    except Exception as e:
        print(f"Database Ingestion Failed: {e}")

Connecting to local PostgreSQL database...
Ingest Layer Success: Transactional insert complete.
